## 서울 따릉이 수요 예측 프로젝트
### 주말, 비 여부를 추가한 Ridge 회귀

**모델**: 다항 특성 + 표준화 + Ridge 회귀

**추가한 전처리**
1. 주말 여부(weekend): 날짜에서 요일을 뽑아 토, 일요일이면 1  
   holiday는 공휴일만 표시해서 주말을 구분하지 못했고, 평일 8시는 평균 1,306대, 주말 8시는 421대로 같은 시간이라도 수요가 크게 달랐기 때문이다.
2. 비 여부(is_rain): 비가 조금이라도 오면 1  
   0.5mm 이하 이슬비에도 평균 대여량이 766대 → 251대로 급감해서 비의 양보다 비가 오는지 여부가 중요했기 때문이다.

alpha 확인: 컬럼을 추가한 뒤 교안과 같은 방법으로 alpha를 0.001 ~ 100으로 비교  
alpha=1이 가장 좋았지만 테스트 R²가 0.861~0.865로 거의 같았다.  
샘플 수(6,772)가 특성 수(740)보다 충분히 많아 과대적합이 크지 않기 때문으로 보인다.

**결과**

| 모델 | MAE | RMSE | R² |
|---|---|---|---|
| 기준 (교안 방식, alpha=0.1) | 173.8 | 264.1 | 0.822 |
| + 주말 여부 + 비 여부 (alpha=1, 최종) | **151.3** | **230.3** | **0.865** |

정리: 주말 여부와 비 여부 컬럼을 추가한 것 만으로 평균 오차가 약 174대 → 151대로 줄어들었고 R²가 0.822 → 0.865로 올랐다.

### 소스코드

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 데이터
df = pd.read_csv('https://raw.githubusercontent.com/tysep16/26_02_ML/refs/heads/main/SeoulBike.csv')

# 전처리
df.columns = ['date', 'rented', 'hour', 'temp', 'humidity', 'wind', 'visibility', 'dew', 'solar',
              'rain', 'snow', 'season', 'holiday', 'functioning']
df = df[df.functioning == 'Yes']
df.holiday = (df.holiday == 'Holiday').astype(int)

# 주말 여부: 날짜에서 요일을 뽑아 토,일요일이면 1
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
df['weekend'] = (df['date'].dt.weekday >= 5).astype(int) # 월요일 0, 일요일 6

# 비 여부: 비가 오면 1
df['is_rain'] = (df['rain'] > 0).astype(int)

df = df.drop(['date', 'functioning'], axis=1) 
y = df['rented']
features = df.drop('rented', axis=1)
features = pd.get_dummies(features, columns=['hour', 'season'], drop_first=True, dtype=int)

# 분할 + 다항 특성 + 표준화
x_train, x_test, y_train, y_test = train_test_split(features, y, test_size=0.2, random_state=42)
poly = PolynomialFeatures(include_bias=False)
x_train = poly.fit_transform(x_train)
x_test = poly.transform(x_test)
ss = StandardScaler()
x_train = ss.fit_transform(x_train)
x_test = ss.transform(x_test)
print('특성 수:', x_train.shape[1])

# alpha 다시 찾기
for alpha in [0.001, 0.01, 0.1, 1, 10, 100]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(x_train, y_train)
    print(alpha, round(ridge.score(x_train, y_train), 3), round(ridge.score(x_test, y_test), 3))

# 가장 좋았던 alpha=1로 최종 모델
ridge = Ridge(alpha=1)
ridge.fit(x_train, y_train)    

# 평가
def evaluate(model, x_test, y_test):
    pred = model.predict(x_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r2 = r2_score(y_test, pred)
    print('MAE :', round(mae, 1), ' RMSE :', round(rmse, 1), ' R2 :', round(r2, 3))
    return pred

pred = evaluate(ridge, x_test, y_test)


특성 수: 740
0.001 0.905 0.862
0.01 0.905 0.863
0.1 0.905 0.863
1 0.904 0.865
10 0.902 0.864
100 0.896 0.861
MAE : 151.3  RMSE : 230.3  R2 : 0.865
